# 北京 PM2.5 线性回归建模

用 DEWP、TEMP、PRES、Iws、Is、Ir 六个气象特征预测 pm2.5。先建线性回归当基线，再用多项式特征加 Ridge 看非线性和过拟合，中间用交叉验证确认结果。

这次先随机划分，按时间切分以后再加。

## 0. 准备数据

In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv("data/PRSA_data_2010.1.1-2014.12.31.csv").dropna(subset=["pm2.5"])
feats = ["DEWP", "TEMP", "PRES", "Iws", "Is", "Ir"]
X = df[feats]
y = df["pm2.5"]
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=42)
print(Xtrain.shape, Xtest.shape)

(33405, 6) (8352, 6)


## 1. 线性回归基线

先建一个最普通的线性回归，训练集 fit、测试集 predict，看 RMSE、MAE、R² 三个指标。

结果 RMSE≈82.3、MAE≈58.7、R²≈0.232。R² 只有 0.23 出头，这六个气象特征只能解释一小部分 PM2.5 的变化，先记下来当基准线。

In [2]:
m = LinearRegression()
m.fit(Xtrain, ytrain)
pred = m.predict(Xtest)
RMSE = mean_squared_error(ytest, pred)
MAE = mean_absolute_error(ytest, pred)
R2 = r2_score(ytest, pred)
print(RMSE, MAE, R2)


6765.370157047045 58.72700674771506 0.23221260486780493


## 2. 五折交叉验证

只切一次训练集测试集，R² 会受切分影响，换个随机种子可能就变。用五折交叉验证（KFold，shuffle，random_state=42）在训练集上跑。

五折 R² 大约是 0.243、0.232、0.233、0.238、0.236，平均 0.2365，和测试集的 0.232 很接近，五次都在 0.23~0.24。线性回归在这份数据上大概就是这个水平。

In [3]:
score =  cross_val_score(m, Xtrain, ytrain, cv=KFold(5, shuffle=True,random_state = 42), scoring='r2')
print(score, score.mean())

[0.24266454 0.23224033 0.23333899 0.23786624 0.23620258] 0.23646253749595667


## 3. 多项式特征 + Ridge 调参

基准线只有 0.23，加多项式特征试试非线性。用 Pipeline 把 PolynomialFeatures、StandardScaler、Ridge 串起来，GridSearchCV 同时搜 degree（1、2、3）和 alpha（0.001~1000）。

最优组合是 degree=3、alpha=0.01，交叉验证 R²≈0.332，测试集 R²≈0.325、RMSE≈77，比基准线好一些。最优 alpha 很小，样本有四万多条而阶数只到 3，模型相对数据量并不复杂，基本不用怎么正则。

In [4]:
pipe = make_pipeline(PolynomialFeatures(), StandardScaler(), Ridge())
params = {'polynomialfeatures__degree':[1,2,3],
          'ridge__alpha':[0.001,0.01,0.1,10,100,1000]
         }
grid = GridSearchCV(pipe,params,cv=KFold(5,shuffle=True,random_state = 42),scoring = 'r2')
grid.fit(Xtrain, ytrain)
print(grid.predict(Xtest))
print(grid.best_params_)
print(grid.best_score_)
print(grid.best_estimator_)

[251.88499232 -48.73521628 103.66094602 ... 119.32793799 122.61448783
  87.11935166]
{'polynomialfeatures__degree': 3, 'ridge__alpha': 0.01}
0.33233480919919084
Pipeline(steps=[('polynomialfeatures', PolynomialFeatures(degree=3)),
                ('standardscaler', StandardScaler()),
                ('ridge', Ridge(alpha=0.01))])


## 4. 小样本 + 高阶：看过拟合

故意只取 300 个样本、阶数开到 6，特征一下扩到八十多个，看过拟合是什么样。对比无正则的 LinearRegression 和 alpha=1、100、10000 的 Ridge，分别记训练集和测试集 R²。

| 模型 | 训练 R² | 测试 R² |
|---|---|---|
| 无正则 LinearRegression | 0.61 | -19630 |
| Ridge alpha=1 | 0.44 | -7.9 |
| Ridge alpha=100 | 0.35 | -0.09 |
| Ridge alpha=10000 | 0.10 | 0.07 |

无正则时训练 R² 有 0.61，测试 R² 却负到近两万，比直接猜平均值还差，模型把训练集里的噪声也背下来了。alpha 调大以后测试分一点点被拉回来，训练分却一直在掉，到 alpha=10000 训练只剩 0.10。

In [5]:
Xs = Xtrain.sample(300, random_state=42)
ys = ytrain.loc[Xs.index]
scores = []
pipe1 = make_pipeline(PolynomialFeatures(6), StandardScaler(),LinearRegression())
pipe1.fit(Xs, ys)
scores.append({'模型':'无正则化的LinearRegression',
              '训练集成绩':pipe1.score(Xs,ys),
              '测试集成绩':pipe1.score(Xtest,ytest),
              })
for i in [1,100,10000]:
    pipe2 = make_pipeline(PolynomialFeatures(6),StandardScaler(),Ridge(i))
    pipe2.fit(Xs, ys)
    scores.append({'模型':f'alpha={i}的Ridge',
              '训练集成绩':pipe2.score(Xs,ys),
              '测试集成绩':pipe2.score(Xtest,ytest)

                  })
print(pd.DataFrame(scores))

                      模型     训练集成绩         测试集成绩
0  无正则化的LinearRegression  0.609963 -19629.670473
1          alpha=1的Ridge  0.438024    -26.459349
2        alpha=100的Ridge  0.345629     -1.066191
3      alpha=10000的Ridge  0.099008      0.072278


## 5. 小结

**交叉验证**　单次划分的分数会受切分影响，五折交叉验证轮流拿五分之一当验证集，结果比较有代表性。Pipeline 把多项式、标准化和模型绑在一起，每一折只用该折的训练数据去 fit 标准化、再套到验证数据上，避免了数据泄漏。

**过拟合和正则**　300 个样本配 6 阶多项式时，无正则模型训练 R² 0.61、测试 R² -19630，是典型过拟合。加大 alpha 后训练 R² 一路降（0.44→0.35→0.10），测试 R² 从 -19630 回到 -7.9、-0.09，alpha=10000 时回到 +0.07，泛化变好；但这时训练 R² 只剩 0.10，又偏欠拟合。

**后面的方向**　目前最好的是 degree=3、alpha=0.01，测试 R² 0.325，比一开始好但仍不高，光靠这六个气象变量不够。接下来想加风向与上一小时的PM2.5、污染是连续的，前一小时浓度对当前影响最大；模型也可以换非线性方法。另外现在是随机划分，等于拿未来预测过去，正式做要按时间切分，等后续学习再尝试